In [56]:
pip install Flask

In [57]:
pip install flask gunicorn openai trulens streamlit plotly


In [58]:
pip install openai

In [59]:
pip install trulens

In [60]:
pip install backports.tarfile


In [38]:
from flask import Flask, render_template, request, jsonify
import os
from openai import OpenAI
import chromadb

from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from trulens.core import TruSession, Select
from trulens.core.feedback import Feedback
from trulens.core.instruments import instrument
from trulens.providers.openai import OpenAI  
from trulens_eval import TruCustomApp
import numpy as np

app = Flask(__name__)

session = TruSession()
session.reset_database()


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


In [7]:
import json

with open("config.json") as f:
    config = json.load(f)

api_key = config["OPENAI_API_KEY"]

In [8]:
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"


NameError: name 'os' is not defined

Get Data

In [41]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

embedding_function = OpenAIEmbeddingFunction(
    api_key=os.environ.get("OPENAI_API_KEY"),
    model_name="text-embedding-ada-002",
)


chroma_client = chromadb.Client()
vector_store = chroma_client.get_or_create_collection(
    name="Washington", embedding_function=embedding_function
)

In [42]:
oai_client = OpenAI()
tru = TruSession()

class RAG:
    @instrument
    def retrieve(self, query: str) -> list:
        """
        Retrieve relevant text from vector store.
        """
        results = vector_store.query(query_texts=query, n_results=4)
        # Flatten the list of lists into a single list
        return [doc for sublist in results["documents"] for doc in sublist]

    @instrument
    def generate_completion(self, query: str, context_str: list) -> str:
        """
        Generate answer from context.
        """
        if len(context_str) == 0:
            return "Sorry, I couldn't find an answer to your question."

        completion = (
            oai_client.chat.completions.create(
                model="gpt-3.5-turbo",
                temperature=0,
                messages=[
                    {
                        "role": "user",
                        "content": f"We have provided context information below. \n"
                        f"---------------------\n"
                        f"{context_str}"
                        f"\n---------------------\n"
                        f"First, say hello and that you're happy to help. \n"
                        f"\n---------------------\n"
                        f"Then, given this information, please answer the question: {query}",
                    }
                ],
            )
            .choices[0]
            .message.content
        )
        if completion:
            return completion
        else:
            return "Did not find an answer."

    @instrument
    def query(self, query: str) -> str:
        context_str = self.retrieve(query=query)
        completion = self.generate_completion(
            query=query, context_str=context_str
        )
        return completion


rag = RAG()

decorating <function RAG.retrieve at 0x7fd67ddf9940>
decorating <function RAG.generate_completion at 0x7fd67ddf9a80>
decorating <function RAG.query at 0x7fd67ddf9b20>
adding method <class '__main__.RAG'> retrieve __main__
adding method <class '__main__.RAG'> generate_completion __main__
adding method <class '__main__.RAG'> query __main__


In [43]:
provider = OpenAI(model_engine="gpt-4")

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [6]:
pip install trulens-dashboard

from trulens.apps.custom import TruCustomApp

tru_rag = TruCustomApp(
    rag,
    app_name="RAG",
    app_version="base",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [46]:
class RAG_from_scratch:
    def __init__(self, vector_store):
        self.vector_store = vector_store

    @instrument
    def retrieve(self, query: str) -> list:
        results = self.vector_store.query(query_texts=query, n_results=2)
        return results["documents"][0]

    @instrument
    def generate_completion(self, query: str, context_str: list) -> str:
        completion = (
            oai_client.chat.completions.create(
                model="gpt-4-turbo-preview	",
                temperature=0,
                messages=[
                    {
                        "role": "user",
                        "content": f"We have provided context information below. \n"
                        f"---------------------\n"
                        f"{context_str}"
                        f"\n---------------------\n"
                        f"Given this information, please answer the question: {query}",
                    }
                ],
            )
            .choices[0]
            .message.content
        )
        return completion

    @instrument
    def query(self, query: str) -> str:
        context_str = self.retrieve(query)
        completion = self.generate_completion(query, context_str)
        return completion

decorating <function RAG_from_scratch.retrieve at 0x7fd67ddf98a0>
decorating <function RAG_from_scratch.generate_completion at 0x7fd67ddf9e40>
decorating <function RAG_from_scratch.query at 0x7fd67ddfbc40>
adding method <class '__main__.RAG_from_scratch'> retrieve __main__
adding method <class '__main__.RAG_from_scratch'> generate_completion __main__
adding method <class '__main__.RAG_from_scratch'> query __main__


In [47]:
def index():
    return render_template("index.html")

@app.route("/process_query", methods=["POST"])
def process_query():
    try:
        data = request.json
        university_info = data.get("university_info")
        query_text = data.get("query")

        if not university_info or not query_text:
            return jsonify({"error": "Both university information and query are required."}), 400

        embedding_function = OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-ada-002")
        chroma_client = chromadb.Client()
        vector_store_local = chroma_client.get_or_create_collection(name="CustomerProductData", embedding_function=embedding_function)
        vector_store_local.add("uni_info", documents=university_info)

        rag = RAG_from_scratch(vector_store_local)
        tru_rag = TruCustomApp(rag, app_id="CS_RAG_v1", feedbacks=[f_groundedness, f_qa_relevance, f_context_relevance])
        with tru_rag as recording:
            result = rag.query(query_text)

        tru.get_leaderboard(app_ids=["CS_RAG_v1"])
        tru.run_dashboard()

        return jsonify({"result": result})
    except Exception as e:
        print(f"Error processing query: {e}")
        return jsonify({"error": str(e)}), 500

In [5]:
pip install "trulens-apps-langchain>=1.0.0"

In [4]:
pip install --upgrade trulens-eval


In [ ]:
if __name__ == "__main__":
    app.run(debug=False, host="0.0.0.0", port=4000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO  [werkzeug] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:3000
 * Running on http://10.250.132.10:3000
INFO  [werkzeug] Press CTRL+C to quit


source venv/bin/activate

In [53]:
session.get_leaderboard()

,,latency,total_cost
app_name,app_version,,


In [24]:
pip install trulens-dashboard

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
from trulens.core import Feedback
from trulens.feedback import GroundTruthAgreement
from trulens.providers.openai import OpenAI as fOpenAI

ground_truth_df = tru.get_ground_truth("test_dataset_new")

f_groundtruth = Feedback(
    GroundTruthAgreement(ground_truth_df, provider=fOpenAI()).agreement_measure,
    name="Ground Truth Semantic Similarity",
).on_input_output()

✅ In Ground Truth Semantic Similarity, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Ground Truth Semantic Similarity, input response will be set to __record__.main_output or `Select.RecordOutput` .
